### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load Dataset

In [2]:
df = pd.read_csv("../dataset/train.csv", dtype={"fullVisitorId": str}, low_memory = False)

In [3]:
df.head()

,channelGrouping,date,device,fullVisitorId,geoNetwork,sessionId,socialEngagementType,totals,trafficSource,visitId,visitNumber,visitStartTime
0,Organic Search,20160902,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",1131660440785968503,"{""continent"": ""Asia"", ""subContinent"": ""Western...",1131660440785968503_1472830385,Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1472830385,1,1472830385
1,Organic Search,20160902,"{""browser"": ""Firefox"", ""browserVersion"": ""not ...",377306020877927890,"{""continent"": ""Oceania"", ""subContinent"": ""Aust...",377306020877927890_1472880147,Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1472880147,1,1472880147
2,Organic Search,20160902,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",3895546263509774583,"{""continent"": ""Europe"", ""subContinent"": ""South...",3895546263509774583_1472865386,Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1472865386,1,1472865386
3,Organic Search,20160902,"{""browser"": ""UC Browser"", ""browserVersion"": ""n...",4763447161404445595,"{""continent"": ""Asia"", ""subContinent"": ""Southea...",4763447161404445595_1472881213,Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1472881213,1,1472881213
4,Organic Search,20160902,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",27294437909732085,"{""continent"": ""Europe"", ""subContinent"": ""North...",27294437909732085_1472822600,Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1472822600,2,1472822600


In [4]:
df.shape

(903653, 12)

In [5]:
df.info(memory_usage = "deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 903653 entries, 0 to 903652
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   channelGrouping       903653 non-null  object
 1   date                  903653 non-null  int64 
 2   device                903653 non-null  object
 3   fullVisitorId         903653 non-null  object
 4   geoNetwork            903653 non-null  object
 5   sessionId             903653 non-null  object
 6   socialEngagementType  903653 non-null  object
 7   totals                903653 non-null  object
 8   trafficSource         903653 non-null  object
 9   visitId               903653 non-null  int64 
 10  visitNumber           903653 non-null  int64 
 11  visitStartTime        903653 non-null  int64 
dtypes: int64(4), object(8)
memory usage: 1.6 GB


In [6]:
(df.memory_usage(deep=True) / 1024**2)

Index                     0.000126
channelGrouping          50.698951
date                      6.894325
device                  675.659032
fullVisitorId            58.497452
geoNetwork              390.546651
sessionId                67.977149
socialEngagementType     59.463555
totals                  101.368099
trafficSource           199.769524
visitId                   6.894325
visitNumber               6.894325
visitStartTime            6.894325
dtype: float64

In [7]:
df[["device","geoNetwork","totals","trafficSource"]].head(1)

,device,geoNetwork,totals,trafficSource
0,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...","{""continent"": ""Asia"", ""subContinent"": ""Western...","{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ..."


In [8]:
print(df.loc[0,"device"])

{"browser": "Chrome", "browserVersion": "not available in demo dataset", "browserSize": "not available in demo dataset", "operatingSystem": "Windows", "operatingSystemVersion": "not available in demo dataset", "isMobile": false, "mobileDeviceBranding": "not available in demo dataset", "mobileDeviceModel": "not available in demo dataset", "mobileInputSelector": "not available in demo dataset", "mobileDeviceInfo": "not available in demo dataset", "mobileDeviceMarketingName": "not available in demo dataset", "flashVersion": "not available in demo dataset", "language": "not available in demo dataset", "screenColors": "not available in demo dataset", "screenResolution": "not available in demo dataset", "deviceCategory": "desktop"}


In [9]:
print(df.loc[0, "geoNetwork"])

{"continent": "Asia", "subContinent": "Western Asia", "country": "Turkey", "region": "Izmir", "metro": "(not set)", "city": "Izmir", "cityId": "not available in demo dataset", "networkDomain": "ttnet.com.tr", "latitude": "not available in demo dataset", "longitude": "not available in demo dataset", "networkLocation": "not available in demo dataset"}


In [10]:
print(df.loc[0,"totals"])

{"visits": "1", "hits": "1", "pageviews": "1", "bounces": "1", "newVisits": "1"}


In [11]:
print(df.loc[0,"trafficSource"])

{"campaign": "(not set)", "source": "google", "medium": "organic", "keyword": "(not provided)", "adwordsClickInfo": {"criteriaParameters": "not available in demo dataset"}}


In [12]:
purchases = df[df["totals"].str.contains("transactionRevenue", regex = False, na = False)]

In [13]:
print(len(purchases))
print(purchases["totals"].iloc[0])

11515
{"visits": "1", "hits": "11", "pageviews": "11", "transactionRevenue": "37860000", "newVisits": "1"}


### Flatten the json data

In [14]:
import json
import gc

In [17]:
FILE_PATH = "../dataset/train.csv"
CHUNK_SIZE = 100000

PLACEHOLDERS = ["(not set)", "(not provided)", "not available in demo dataset"]

In [20]:
def flatten_chunk(chunk):
    device  = chunk["device"].apply(json.loads)
    geo     = chunk["geoNetwork"].apply(json.loads)
    totals  = chunk["totals"].apply(json.loads)
    traffic = chunk["trafficSource"].apply(json.loads)

    out = pd.DataFrame({
        "fullVisitorId"   : chunk["fullVisitorId"].values,
        "visitId"         : chunk["visitId"].values,
        "visitNumber"     : chunk["visitNumber"].values,
        "date"            : chunk["date"].values,
        "channelGrouping" : chunk["channelGrouping"].values,

        # device
        "deviceCategory"  : [d.get("deviceCategory") for d in device],
        "browser"         : [d.get("browser") for d in device],
        "operatingSystem" : [d.get("operatingSystem") for d in device],
        "isMobile"        : [d.get("isMobile", False) for d in device],

        # geoNetwork
        "continent"       : [g.get("continent") for g in geo],
        "subContinent"    : [g.get("subContinent") for g in geo],
        "country"         : [g.get("country") for g in geo],

        # Totals
        "pageviews"       : [t.get("pageviews", 0) for t in totals],
        "bounces"         : [t.get("bounces", 0) for t in totals],
        "hits"            : [t.get("hits", 0) for t in totals],
        "newVisits"       : [t.get("newVisits", 0) for t in totals],
        "transactionRevenue" : [t.get("transactionRevenue", 0) for t in totals],

        #trafficSource
        "source"          : [t.get("source") for t in traffic],
        "medium"          : [t.get("medium") for t in traffic],
        "campaign"        : [t.get("campaign") for t in traffic]
    })

    # text -> numbers
    numeric_cols = ["pageviews", "bounces", "hits", "newVisits", "transactionRevenue"]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors = "coerce").fillna(0)

    # revenue in real dollars
    out["Revenue"] = out["transactionRevenue"] / 1_000_000
    out = out.drop("transactionRevenue", axis = 1)

    # downcast to save memory
    for col in ["pageviews", "bounces", "hits", "newVisits"]:
        out[col] = out[col].astype("int32")

    # dates 
    out["date"] = pd.to_datetime(out["date"].astype("str"), format = "%Y%m%d")
    out["YearMonth"] = out["date"].dt.to_period("M").astype("str")

    # placeholder text --> unknown
    text_cols = ["deviceCategory","browser", "operatingSystem", "continent", "subContinent",
                 "country", "source", "medium", "campaign"]
    out[text_cols] = out[text_cols].replace(PLACEHOLDERS, "Unknown").fillna("Unknown")

    # funnel flags
    out["IsCustomer"] = (out["Revenue"] > 0).astype("int8")
    out["IsLead"]     = (out["bounces"] == 0).astype("int8")
    out["FunnelStage"] = np.select(
        [out["IsCustomer"] == 1, out["IsLead"] == 1],
        ["Customer", "Lead"],
        default="Visitor Only"
    )
    return out

In [36]:
chunks = []
reader = pd.read_csv(FILE_PATH, dtype = {"fullVisitorId" : str}, chunksize = CHUNK_SIZE)

for i, chunk in enumerate(reader):
    chunks.append(flatten_chunk(chunk))
    print(f"chunk {i+1} done | rows so far: {sum(len(c) for c in chunks):,}")
    del chunk
    gc.collect()

chunk 1 done | rows so far: 100,000
chunk 2 done | rows so far: 200,000
chunk 3 done | rows so far: 300,000
chunk 4 done | rows so far: 400,000
chunk 5 done | rows so far: 500,000
chunk 6 done | rows so far: 600,000
chunk 7 done | rows so far: 700,000
chunk 8 done | rows so far: 800,000
chunk 9 done | rows so far: 900,000
chunk 10 done | rows so far: 903,653


In [39]:
df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

559

In [43]:
df.to_csv("../dataset/ga_sessions_flat.csv", index=False)

In [40]:
df.head()

,fullVisitorId,visitId,visitNumber,date,channelGrouping,deviceCategory,browser,operatingSystem,isMobile,continent,...,hits,newVisits,source,medium,campaign,Revenue,YearMonth,IsCustomer,IsLead,FunnelStage
0,1131660440785968503,1472830385,1,2016-09-02,Organic Search,desktop,Chrome,Windows,False,Asia,...,1,1,google,organic,Unknown,0.0,2016-09,0,0,Visitor Only
1,377306020877927890,1472880147,1,2016-09-02,Organic Search,desktop,Firefox,Macintosh,False,Oceania,...,1,1,google,organic,Unknown,0.0,2016-09,0,0,Visitor Only
2,3895546263509774583,1472865386,1,2016-09-02,Organic Search,desktop,Chrome,Windows,False,Europe,...,1,1,google,organic,Unknown,0.0,2016-09,0,0,Visitor Only
3,4763447161404445595,1472881213,1,2016-09-02,Organic Search,desktop,UC Browser,Linux,False,Asia,...,1,1,google,organic,Unknown,0.0,2016-09,0,0,Visitor Only
4,27294437909732085,1472822600,2,2016-09-02,Organic Search,mobile,Chrome,Android,True,Europe,...,1,0,google,organic,Unknown,0.0,2016-09,0,0,Visitor Only


In [41]:
df.shape

(903653, 24)

In [44]:
df.info(memory_usage = "deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 903653 entries, 0 to 903652
Data columns (total 24 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   fullVisitorId    903653 non-null  object        
 1   visitId          903653 non-null  int64         
 2   visitNumber      903653 non-null  int64         
 3   date             903653 non-null  datetime64[ns]
 4   channelGrouping  903653 non-null  object        
 5   deviceCategory   903653 non-null  object        
 6   browser          903653 non-null  object        
 7   operatingSystem  903653 non-null  object        
 8   isMobile         903653 non-null  bool          
 9   continent        903653 non-null  object        
 10  subContinent     903653 non-null  object        
 11  country          903653 non-null  object        
 12  pageviews        903653 non-null  int32         
 13  bounces          903653 non-null  int32         
 14  hits             903

In [45]:
cat_cols = ["channelGrouping", "deviceCategory", "browser", "operatingSystem",
            "continent", "subContinent", "country", "source", "medium",
            "campaign", "FunnelStage"]
for c in cat_cols:
    df[c] = df[c].astype("category")

df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 903653 entries, 0 to 903652
Data columns (total 24 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   fullVisitorId    903653 non-null  object        
 1   visitId          903653 non-null  int64         
 2   visitNumber      903653 non-null  int64         
 3   date             903653 non-null  datetime64[ns]
 4   channelGrouping  903653 non-null  category      
 5   deviceCategory   903653 non-null  category      
 6   browser          903653 non-null  category      
 7   operatingSystem  903653 non-null  category      
 8   isMobile         903653 non-null  bool          
 9   continent        903653 non-null  category      
 10  subContinent     903653 non-null  category      
 11  country          903653 non-null  category      
 12  pageviews        903653 non-null  int32         
 13  bounces          903653 non-null  int32         
 14  hits             903

In [48]:
print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Visitors:", len(df))
print("Leads:", df["IsLead"].sum())
print("Customers:", df["IsCustomer"].sum())
print("Customers failing the Lead rule:", ((df["IsCustomer"] == 1) & (df["IsLead"] == 0)).sum())
print("Total revenue: $", round(df["Revenue"].sum(), 2))
print(df.isnull().sum())

Shape: (903653, 24)
Date range: 2016-08-01 00:00:00 to 2017-08-01 00:00:00
Visitors: 903653
Leads: 453023
Customers: 11515
Customers failing the Lead rule: 0
Total revenue: $ 1540071.24
fullVisitorId      0
visitId            0
visitNumber        0
date               0
channelGrouping    0
deviceCategory     0
browser            0
operatingSystem    0
isMobile           0
continent          0
subContinent       0
country            0
pageviews          0
bounces            0
hits               0
newVisits          0
source             0
medium             0
campaign           0
Revenue            0
YearMonth          0
IsCustomer         0
IsLead             0
FunnelStage        0
dtype: int64
